# Week 2 on Colab — train your GPT

This is the first run in the course that needs a GPU, and the first that is long enough for a session to die in the middle of it.

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Do not start this until `pytest week-02` is green on your laptop and pushed.

## 1. Get your code

In [ ]:
import os

REPO_URL = "..."   # the address of YOUR repo

if not os.path.exists("course") and os.path.basename(os.getcwd()) != "course":
    !git clone {REPO_URL} course
if os.path.basename(os.getcwd()) != "course":
    %cd course
!pip install -q -e . pytest

## 2. Somewhere that survives

Colab's own disk is wiped when the session ends. Your Drive is not. This asks for permission once.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

os.makedirs("/content/drive/MyDrive/tunelab", exist_ok=True)
CHECKPOINT = "/content/drive/MyDrive/tunelab/week-02.pt"
print("checkpoint already there:", os.path.exists(CHECKPOINT))

## 3. Is there a GPU?

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then start again from the top."
print(torch.cuda.get_device_name(0))

## 4. Does your model work on it?

Your tests ran on a CPU, and there is one kind of bug a CPU cannot show: a tensor created inside `forward` that stays behind on the CPU when the model moves.

If this cell fails with **`Expected all tensors to be on the same device`**, it is one of two things, and Day 4's README names both: the positions in `Embedder`, or the mask in `Head`. Fix it on your laptop, push, and start this notebook again.

In [ ]:
import importlib.util
from pathlib import Path

from tunelab.gpt import GPT

spec = importlib.util.spec_from_file_location("train_gpt", "week-02/milestone/train_gpt.py")
train_gpt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_gpt)

model = GPT(vocab_size=65, **train_gpt.BIG).to("cuda")
idx = torch.randint(0, 65, (2, 128), device="cuda")
targets = torch.randint(0, 65, (2, 128), device="cuda")
logits, loss = model(idx, targets)
out = model.generate(idx[:, :1], max_new_tokens=5)
assert logits.is_cuda and out.is_cuda
print("forward and generate both work on the GPU.")
print("starting loss", round(loss.item(), 2), "- should be close to ln(65) = 4.17")
del model

## 5. Train

3000 steps, saved to your Drive every 500. The cell prints one line each time a save completes.

**You must kill this run once, on purpose.** When it has printed its line for **step 1000**, choose **Runtime → Interrupt execution**. Then run this same cell again. It should say it is carrying on from step 1000 or later, and it must not start again from zero. If it starts from zero, your checkpointing has a bug: fix it on your laptop, push, and start over.

If Colab disconnects on you by itself, that counts. Reconnect, run the cells above again, then this one.

In [ ]:
from tunelab.checkpoint import load

done, _ = load(CHECKPOINT, GPT(vocab_size=65, **train_gpt.BIG))
print(f"carrying on from step {done}" if done else "starting from step 0")

text = Path("data/tiny.txt").read_text()

# run() carries on from its checkpoint, so asking for a little more each time
# gives a line of progress at every save.
for target in range(500, 3001, 500):
    if target <= done:
        continue
    result = train_gpt.run(
        text, train_gpt.BIG, steps=target, path=CHECKPOINT,
        batch_size=64, lr=1e-3, chunk=500, device="cuda",
    )
    last = result["history"][-1]
    print(f'step {target:5d} saved   train {last["train"]:.3f}   val {last["val"]:.3f}')

if done >= 3000:
    result = train_gpt.run(text, train_gpt.BIG, steps=3000, path=CHECKPOINT, batch_size=64, lr=1e-3, device="cuda")
print(f'{result["params"]:,} parameters, final validation loss {result["val_loss"]:.3f}')

## 6. Look at it

In [ ]:
import matplotlib.pyplot as plt

steps = [row["step"] for row in result["history"]]
plt.plot(steps, [row["train"] for row in result["history"]], label="train")
plt.plot(steps, [row["val"] for row in result["history"]], label="val")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

print(result["sample"])

## 7. Write it down

A row in `logs/run-log.md`: parameters, steps, final train and validation loss, and the step you killed it at. Paste the sample above into `week-02/milestone/sample.txt`. Commit both from your laptop.